In [ ]:
# ============================================================================
# 03_A_PyTorch.ipynb - Cell 1: RUN
# Generalizes the PyTorch star-detector (previously one-off, V CrA only) to
# every RCB star folder under BASE_DATA_DIR, mirroring 02_A_SISaP.ipynb's
# fully-automated multi-star architecture: no manual paradigm labeling, no
# interactive widgets -- everything below runs unattended per star.
#
# Per star this does, all self-contained:
#   1. Trains a UNet star-detector on that star's OWN plates. Training
#      labels are pseudo-labels only (DAOStarFinder positions on plates
#      that pass a defect-based quality gate) -- no human-labeled paradigm
#      plate, so this can run across every star without stopping to ask
#      for input.
#   2. Runs the same APASS-calibrated aperture photometry pipeline as
#      02_A_SISaP.ipynb, with one addition: a new target-location tier
#      that asks the trained model where the target is, used as a
#      fallback whenever the target isn't already an APASS-catalogued
#      star. This is what lets a deeply-faded RCB star (invisible to a
#      naive threshold check) still get located and measured.
#   3. Saves photometry_results.csv / lightcurve.csv / reference_photometry.csv
#      plus the trained model and training images/masks, all under each
#      star's own <star>/03_A/ folder.
# No plotting here -- see Cell 2 for the This-Pipeline-vs-DASCH comparison,
# which reads these output files back from disk (works even after a
# kernel restart, any time after this cell has produced them).
# ============================================================================

from pathlib import Path
import shelve
import time
import copy
import traceback
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
from scipy import ndimage
from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.stats import SigmaClip, sigma_clipped_stats
import astropy.units as u
from photutils.detection import DAOStarFinder, find_peaks
from photutils.aperture import (CircularAperture, CircularAnnulus,
                                 ApertureStats, aperture_photometry)
from photutils.centroids import centroid_sources, centroid_com
from photutils.background import Background2D, MedianBackground
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import ipywidgets as widgets
from IPython.display import display
import urllib.request
import urllib.parse

# ============================================================================
# PATHS
# ============================================================================
BASE_DATA_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data")

star_dirs = sorted([d for d in BASE_DATA_DIR.iterdir()
                     if d.is_dir() and (d / 'cutouts').is_dir()])
print(f"Found {len(star_dirs)} star folders with a cutouts/ directory:")
for d in star_dirs:
    n_fits = len(list((d / 'cutouts').glob('*.fits')))
    print(f"  {d.name}: {n_fits} plates")

SHARED_CACHE_DIR = BASE_DATA_DIR / '_shared_cache_03_A'
SHARED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
apass_cache = shelve.open(str(SHARED_CACHE_DIR / 'apass_cache_shelf'), flag='c', writeback=False)
print(f"\nLoaded {len(apass_cache)} cached APASS field queries (shared across all stars)")

MANUAL_TARGET_COORDS = {
    'V_CrA': SkyCoord(ra=281.884623417 * u.deg, dec=-38.158974417 * u.deg),
}

TARGET_COORD_CACHE_PATH = SHARED_CACHE_DIR / 'target_coords.csv'

def resolve_target_coord(star_dir):
    """Get (and cache) the target coordinate for a star folder."""
    name = star_dir.name
    if name in MANUAL_TARGET_COORDS:
        return MANUAL_TARGET_COORDS[name], 'manual'
    if TARGET_COORD_CACHE_PATH.exists():
        cached = pd.read_csv(TARGET_COORD_CACHE_PATH)
        match = cached[cached['folder_name'] == name]
        if len(match) > 0:
            row = match.iloc[0]
            return SkyCoord(ra=row['ra_deg'] * u.deg, dec=row['dec_deg'] * u.deg), 'cached'
    query_name = name.replace('_', ' ')
    try:
        coord = SkyCoord.from_name(query_name)
        row = pd.DataFrame([{'folder_name': name, 'query_name': query_name,
                              'ra_deg': coord.ra.deg, 'dec_deg': coord.dec.deg}])
        if TARGET_COORD_CACHE_PATH.exists():
            row.to_csv(TARGET_COORD_CACHE_PATH, mode='a', header=False, index=False)
        else:
            row.to_csv(TARGET_COORD_CACHE_PATH, index=False)
        return coord, 'resolved'
    except Exception as e:
        return None, f'FAILED: {e}'

# ============================================================================
# PHOTOMETRY PARAMETERS (identical to 02_A_SISaP.ipynb, unchanged)
# ============================================================================
DEFAULT_APERTURE = 6
ANNULUS_INNER = 15
ANNULUS_OUTER = 20
CENTROID_BOX = 21
CENTROID_MAX_DRIFT = 5
SIGNIF_THRESHOLD = 5.0
SATURATION_FRACTION = 0.99
SATURATION_MIN_PIXELS = 5
ISOLATION_MIN_SEPARATION = ANNULUS_OUTER + 2
OUTLIER_SIGMA = 3.0
REF_STARS_MAX_PER_PLATE = 150

MIN_REF_STARS = 25
LOCAL_HALF_WINDOW = 1.5
MIN_REF_STARS_LOCAL = 8
LOCAL_WINDOW_GROWTH = 1.6

MAX_CALIBRATION_RMS = 0.5

PIPELINE_VERSION = 1
MIN_APASS_STARS_QUERY = 10

# ============================================================================
# PYTORCH PARAMETERS
# ============================================================================
DAO_FWHM = 3.0
DAO_THRESHOLD_SIGMA = 5.0
DAO_MIN_SOURCES = 5
FINDPEAKS_BOX_SIZE = 11

MASK_BOX_HALF = 8            # half-width of the training mask box drawn at each pseudo-label position
TRAIN_VAL_SPLIT = 0.15
TRAIN_MAX_EPOCHS = 30
TRAIN_EARLY_STOP_PATIENCE = 6
TRAIN_BATCH_SIZE = 1         # 1, not 2 -- keeps this robust to stars whose plates aren't all the same pixel size
MIN_TRAINING_PLATES = 4      # below this many usable plates, skip training for this star entirely

PRED_MASK_THRESHOLD = 0.5
MIN_BLOB_AREA = 4
PYTORCH_MATCH_TOLERANCE_PX = 15  # how close a model-predicted blob must be to the expected WCS position to count

MODEL_VERSION = 1  # bump to force every star's model to be retrained

# ============================================================================
# CORE PHOTOMETRY FUNCTIONS (identical to 02_A_SISaP.ipynb, unchanged)
# ============================================================================

def subtract_background_2d(data, box_size=50):
    try:
        bkg = Background2D(data, (box_size, box_size), filter_size=(3, 3),
                            bkg_estimator=MedianBackground())
        return data - bkg.background, bkg.background_rms
    except Exception:
        med = np.nanmedian(data)
        return data - med, np.full_like(data, np.nanstd(data))

def refine_centroids(data, x, y, box_size=CENTROID_BOX, max_drift=CENTROID_MAX_DRIFT):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) == 0:
        return x, y
    try:
        x_ref, y_ref = centroid_sources(data, x, y, box_size=box_size,
                                         centroid_func=centroid_com)
        good = np.isfinite(x_ref) & np.isfinite(y_ref)
        drift = np.hypot(x_ref - x, y_ref - y)
        good &= drift < max_drift
        x_ref[~good] = x[~good]
        y_ref[~good] = y[~good]
        return x_ref, y_ref
    except Exception:
        return x, y

def filter_isolated_stars(x, y, min_separation=ISOLATION_MIN_SEPARATION):
    n = len(x)
    if n < 2:
        return np.ones(n, dtype=bool)
    isolated = np.ones(n, dtype=bool)
    for i in range(n):
        d = np.hypot(x - x[i], y - y[i])
        d[i] = np.inf
        if np.any(d < min_separation):
            isolated[i] = False
    return isolated

def measure_aperture_photometry(data, x, y, radius=DEFAULT_APERTURE,
                                 annulus_inner=ANNULUS_INNER, annulus_outer=ANNULUS_OUTER):
    n = len(x)
    if n == 0:
        return np.array([]), np.array([]), np.array([])
    positions = list(zip(x, y))
    aperture = CircularAperture(positions, r=radius)
    annulus = CircularAnnulus(positions, r_in=annulus_inner, r_out=annulus_outer)
    ann_stats = ApertureStats(data, annulus, sigma_clip=SigmaClip(sigma=3.0))
    bkg_median = np.nan_to_num(ann_stats.median, nan=0.0)
    bkg_std = np.nan_to_num(ann_stats.std, nan=0.0)
    phot = aperture_photometry(data, aperture)
    flux = phot['aperture_sum'].value - bkg_median * (np.pi * radius**2)
    return flux, bkg_median, bkg_std

def detect_saturation(data, x, y, radius=DEFAULT_APERTURE,
                       sat_fraction=SATURATION_FRACTION, min_pixels=SATURATION_MIN_PIXELS):
    n = len(x)
    is_sat = np.zeros(n, dtype=bool)
    data_max = np.nanmax(data)
    if data_max <= 0:
        return is_sat
    positions = list(zip(x, y))
    aperture = CircularAperture(positions, r=radius)
    aperture_masks = aperture.to_mask(method='center')
    for i in range(n):
        mask = aperture_masks[i]
        cut = mask.multiply(data)
        if cut is None:
            continue
        star_pixels = cut[mask.data > 0]
        if len(star_pixels) == 0:
            continue
        near_max = np.sum(star_pixels >= sat_fraction * data_max)
        is_sat[i] = near_max >= min_pixels
    return is_sat

def calibrate_global(inst_mags, apass_b, outlier_sigma=OUTLIER_SIGMA, min_stars=MIN_REF_STARS):
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b)
    if mask.sum() < min_stars:
        return None
    x = apass_b[mask]
    y = inst_mags[mask]
    for _ in range(3):
        try:
            coeffs = np.polyfit(x, y, 2)
            residuals = y - np.polyval(coeffs, x)
            rms = np.sqrt(np.mean(residuals**2))
            good = np.abs(residuals) < outlier_sigma * rms
            if good.sum() < min_stars:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < min_stars:
        return None
    coeffs = np.polyfit(x, y, 2)
    residuals = y - np.polyval(coeffs, x)
    rms = np.sqrt(np.mean(residuals**2))
    return {'coeffs': coeffs, 'rms': rms, 'n_used': len(x),
            'b_min': float(np.min(x)), 'b_max': float(np.max(x))}

def calibrate_local(inst_mags, apass_b, target_b_estimate,
                     half_window=LOCAL_HALF_WINDOW, min_stars=MIN_REF_STARS_LOCAL,
                     growth=LOCAL_WINDOW_GROWTH, outlier_sigma=OUTLIER_SIGMA):
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b)
    x_all, y_all = apass_b[mask], inst_mags[mask]
    if len(x_all) < min_stars:
        return None
    window = half_window
    local_mask = np.abs(x_all - target_b_estimate) <= window
    for _ in range(5):
        if local_mask.sum() >= min_stars:
            break
        window *= growth
        local_mask = np.abs(x_all - target_b_estimate) <= window
    if local_mask.sum() < min_stars:
        return None
    x, y = x_all[local_mask], y_all[local_mask]
    for _ in range(3):
        try:
            coeffs = np.polyfit(x, y, 1)
            residuals = y - np.polyval(coeffs, x)
            rms = np.sqrt(np.mean(residuals**2))
            good = np.abs(residuals) < outlier_sigma * rms
            if good.sum() < min_stars:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < min_stars:
        return None
    coeffs = np.polyfit(x, y, 1)
    residuals = y - np.polyval(coeffs, x)
    rms = np.sqrt(np.mean(residuals**2))
    return {'coeffs': coeffs, 'rms': rms, 'n_used': len(x),
            'window_used': float(window),
            'b_min': float(np.min(x)), 'b_max': float(np.max(x))}

def invert_calibration(calibration, t_inst_mag):
    coeffs = calibration['coeffs']
    poly = list(coeffs)
    poly[-1] = poly[-1] - t_inst_mag
    roots = np.roots(poly)
    real_roots = roots[np.abs(roots.imag) < 1e-6].real
    if len(real_roots) == 0:
        return None
    lo, hi = calibration['b_min'], calibration['b_max']
    in_range = real_roots[(real_roots >= lo - 2) & (real_roots <= hi + 2)]
    candidates = in_range if len(in_range) > 0 else real_roots
    best = min(candidates, key=lambda r: abs(np.polyval(coeffs, r) - t_inst_mag))
    return float(best)

def query_apass(ra_c, dec_c, radius_arcsec, max_rows=3000):
    params = {
        '-source': 'II/336/apass9',
        '-c': f"{ra_c} {dec_c}",
        '-c.rs': radius_arcsec,
        '-out': 'RAJ2000,DEJ2000,Bmag,e_Bmag',
        '-out.max': max_rows,
    }
    url = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv?" + urllib.parse.urlencode(params)
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as response:
                return response.read().decode('utf-8')
        except Exception:
            time.sleep(2 ** attempt)
    return None

def parse_apass_tsv(text):
    rows = []
    for line in text.split('\n'):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        parts = line.split('\t')
        if len(parts) < 4:
            continue
        try:
            ra, dec, bmag, e_bmag = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
            rows.append((ra, dec, bmag, e_bmag))
        except (ValueError, IndexError):
            continue
    if rows and not (0 < rows[0][2] < 25):
        rows = rows[1:]
    return rows

def get_field_catalog(ra_c, dec_c, radius_arcsec):
    key = f"{round(ra_c, 3)}_{round(dec_c, 3)}"
    if key in apass_cache:
        return apass_cache[key]
    raw = query_apass(ra_c, dec_c, radius_arcsec)
    rows = []
    if raw:
        rows = parse_apass_tsv(raw)
        rows = [r for r in rows if 0 < r[2] < 20 and r[3] < 0.5]
    apass_cache[key] = rows
    return rows

def get_plate_jd(header):
    if 'JD-OBS' in header:
        return float(header['JD-OBS'])
    if 'MJD-OBS' in header:
        return float(header['MJD-OBS']) + 2400000.5
    for key in ('DATE-OBS', 'DATE'):
        if key in header:
            try:
                return Time(header[key]).jd
            except Exception:
                continue
    return np.nan

# ============================================================================
# LIGHTWEIGHT DEFECT DETECTION (for pseudo-label plate selection only --
# trimmed from plate_scan_utils.py's version, same core checks, no scratch
# angle detection since that's not needed to just gate "is this plate too
# messy to learn from")
# ============================================================================

def detect_plate_defects(data):
    h, w = data.shape
    lo, hi = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    if hi == lo:
        return {"n_saturation": 0, "dead_zone_fraction": 0.0, "saturation_area_fraction": 0.0, "edge": False}
    norm = np.clip((data - lo) / (hi - lo), 0, 1)

    n_saturation = 0
    saturation_area_fraction = 0.0
    try:
        sat_thresh = np.percentile(norm, 99.9)
        sat_mask = (norm >= sat_thresh).astype(np.uint8)
        sat_mask = ndimage.binary_dilation(sat_mask, iterations=2).astype(np.uint8)
        saturation_area_fraction = float(sat_mask.sum()) / float(h * w)
        labeled, n_obj = ndimage.label(sat_mask)
        for obj_id in range(1, n_obj + 1):
            if (labeled == obj_id).sum() >= 200:
                n_saturation += 1
    except Exception:
        pass

    dead_zone_fraction = 0.0
    try:
        raw_min, raw_max = np.nanmin(data), np.nanmax(data)
        tol = max(1.0, 0.02 * (raw_max - raw_min))
        dead_mask = (data <= raw_min + tol).astype(np.uint8)
        labeled_dead, n_dead_obj = ndimage.label(dead_mask)
        if n_dead_obj > 0:
            sizes = ndimage.sum(dead_mask, labeled_dead, index=range(1, n_dead_obj + 1))
            dead_zone_fraction = float(np.max(sizes)) / float(h * w) if len(sizes) else 0.0
    except Exception:
        pass

    edge = False
    try:
        border = max(20, int(min(h, w) * 0.05))
        center_med = np.nanmedian(norm[border:h-border, border:w-border])
        center_std = np.nanstd(norm[border:h-border, border:w-border])
        for strip in (norm[:border, :], norm[h-border:, :], norm[:, :border], norm[:, w-border:]):
            if abs(np.nanmedian(strip) - center_med) > 3 * center_std:
                edge = True
                break
    except Exception:
        pass

    return {"n_saturation": n_saturation, "dead_zone_fraction": dead_zone_fraction,
            "saturation_area_fraction": saturation_area_fraction, "edge": edge}

def plate_ok_for_training(defects, n_sources):
    """True if this plate is clean enough to trust its DAOStarFinder
    detections as pseudo-labels."""
    if defects["dead_zone_fraction"] >= 0.08:
        return False
    if defects["saturation_area_fraction"] >= 0.05:
        return False
    if defects["n_saturation"] >= 8:
        return False
    if defects["edge"] and n_sources < 3:
        return False
    return True

def detect_sources_dao(fits_path):
    """DAOStarFinder with a find_peaks fallback -- same detector as
    02_B's plate_scan_utils.detect_sources(), trimmed to just positions
    (no aperture_flux column needed here)."""
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    mean, median, std = sigma_clipped_stats(data, sigma=3.0)
    data_sub = data - median
    try:
        daofind = DAOStarFinder(fwhm=DAO_FWHM, threshold=DAO_THRESHOLD_SIGMA * std, sharpness_range=(0.2, 2.0))
        sources = daofind(data_sub)
    except Exception:
        sources = None
    if sources is None or len(sources) < DAO_MIN_SOURCES:
        sources = find_peaks(data_sub, threshold=DAO_THRESHOLD_SIGMA * std, box_size=FINDPEAKS_BOX_SIZE)
    if sources is None or len(sources) == 0:
        return np.array([]), np.array([]), data
    if "x_centroid" in sources.colnames:
        xs, ys = np.array(sources["x_centroid"]), np.array(sources["y_centroid"])
    else:
        xs, ys = np.array(sources["x_peak"]), np.array(sources["y_peak"])
    return xs.astype(float), ys.astype(float), data

# ============================================================================
# UNET MODEL (identical architecture to 02_B_PyTorch_Algorithm.ipynb)
# ============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def crop_to_match(x, ref):
    _, _, h, w = ref.shape
    return x[:, :, :h, :w]

def pad_to_multiple(img, multiple=16):
    _, h, w = img.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    return F.pad(img, (0, pad_w, 0, pad_h))

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU())
        self.enc1, self.enc2, self.enc3 = block(1, 32), block(32, 64), block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(128, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, stride=2), block(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, stride=2), block(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, stride=2), block(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(m), crop_to_match(e3, self.up3(m))], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), crop_to_match(e2, self.up2(d3))], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), crop_to_match(e1, self.up1(d2))], dim=1))
        return torch.sigmoid(self.out(d1))

class StarSegDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths, self.mask_paths = image_paths, mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        img = pad_to_multiple(torch.tensor(img).unsqueeze(0))
        mask = pad_to_multiple(torch.tensor(mask).unsqueeze(0))
        return img, mask

def _points_to_mask(shape, xs, ys, box_half=MASK_BOX_HALF):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    for x, y in zip(xs, ys):
        x0, x1 = max(0, int(x - box_half)), min(w, int(x + box_half))
        y0, y1 = max(0, int(y - box_half)), min(h, int(y + box_half))
        mask[y0:y1, x0:x1] = 255
    return mask

def _save_image_mask_pair(data, xs, ys, stem, img_dir, mask_dir):
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)
    mask = _points_to_mask(img_8bit.shape, xs, ys)
    img_path = img_dir / (stem + ".png")
    mask_path = mask_dir / (stem + "_mask.png")
    cv2.imwrite(str(img_path), img_8bit)
    cv2.imwrite(str(mask_path), mask)
    return img_path, mask_path

def build_dataset_and_train(star_dir, cutouts, status_label):
    """Fully automated: no paradigm plate, no human labeling. Every plate
    clean enough per plate_ok_for_training() contributes its DAOStarFinder
    detections as pseudo-labels. Returns (model, loss_history_dict) or
    (None, None) if too few usable plates."""
    star_output_dir = star_dir / '03_A'
    img_dir = star_output_dir / 'images'
    mask_dir = star_output_dir / 'masks'
    img_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    image_paths, mask_paths = [], []
    n_skipped = 0
    for i, f in enumerate(cutouts):
        status_label.value = f"{star_dir.name}: building training pairs ({i+1}/{len(cutouts)}) -- {f.name}"
        try:
            xs, ys, data = detect_sources_dao(f)
            if len(xs) == 0:
                n_skipped += 1
                continue
            defects = detect_plate_defects(data)
            if not plate_ok_for_training(defects, len(xs)):
                n_skipped += 1
                continue
            ip, mp = _save_image_mask_pair(data, xs, ys, f.stem, img_dir, mask_dir)
            image_paths.append(ip)
            mask_paths.append(mp)
        except Exception:
            n_skipped += 1
            continue

    print(f"  {star_dir.name}: {len(image_paths)} plate(s) usable for training, {n_skipped} skipped (too messy or empty).")

    if len(image_paths) < MIN_TRAINING_PLATES:
        print(f"  {star_dir.name}: too few usable plates to train a model -- PyTorch target-location tier will be skipped for this star.")
        return None, None

    combined = list(zip(image_paths, mask_paths))
    np.random.shuffle(combined)
    n_val = max(1, int(len(combined) * TRAIN_VAL_SPLIT))
    val_pairs, train_pairs = combined[:n_val], combined[n_val:]

    train_ds = StarSegDataset([p[0] for p in train_pairs], [p[1] for p in train_pairs])
    val_ds = StarSegDataset([p[0] for p in val_pairs], [p[1] for p in val_pairs])
    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=TRAIN_BATCH_SIZE)

    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCELoss()

    train_loss_history, val_loss_history = [], []
    best_val_loss = float("inf")
    best_state_dict = None
    epochs_without_improvement = 0
    stopped_early_at = None

    for epoch in range(TRAIN_MAX_EPOCHS):
        model.train()
        train_loss_sum = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, masks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_loss_sum += loss.item()
        train_loss_avg = train_loss_sum / max(len(train_loader), 1)

        model.eval()
        val_loss_sum = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                val_loss_sum += loss_fn(model(imgs), masks).item()
        val_loss_avg = val_loss_sum / max(len(val_loader), 1)

        train_loss_history.append(train_loss_avg)
        val_loss_history.append(val_loss_avg)
        status_label.value = f"{star_dir.name}: training epoch {epoch+1}/{TRAIN_MAX_EPOCHS} -- train={train_loss_avg:.4f} val={val_loss_avg:.4f}"

        if val_loss_avg < best_val_loss - 1e-5:
            best_val_loss = val_loss_avg
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= TRAIN_EARLY_STOP_PATIENCE:
                stopped_early_at = epoch + 1
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    model_path = star_output_dir / 'pytorch_model.pth'
    torch.save(model.state_dict(), model_path)
    with open(star_output_dir / 'model_version.txt', 'w') as vf:
        vf.write(str(MODEL_VERSION))

    stop_note = f"stopped early at epoch {stopped_early_at}" if stopped_early_at else f"ran all {TRAIN_MAX_EPOCHS} epochs"
    print(f"  {star_dir.name}: model trained ({stop_note}, best val loss {best_val_loss:.4f}). Saved to {model_path}")

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 3.5))
        epochs_range = list(range(1, len(train_loss_history) + 1))
        ax.plot(epochs_range, train_loss_history, marker="o", markersize=3, label="Train Loss")
        ax.plot(epochs_range, val_loss_history, marker="o", markersize=3, label="Val Loss")
        ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss (avg/batch)")
        ax.set_title(f"{star_dir.name} training progress")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(star_output_dir / 'training_loss.png', dpi=100)
        plt.close(fig)
    except Exception:
        pass

    return model, {"train_loss": train_loss_history, "val_loss": val_loss_history, "best_val_loss": best_val_loss}

def load_existing_model(star_dir):
    """Reuse a previously-trained model for this star if one exists at
    the current MODEL_VERSION, skipping retraining."""
    star_output_dir = star_dir / '03_A'
    model_path = star_output_dir / 'pytorch_model.pth'
    version_path = star_output_dir / 'model_version.txt'
    if not model_path.exists() or not version_path.exists():
        return None
    try:
        with open(version_path) as vf:
            saved_version = int(vf.read().strip())
        if saved_version != MODEL_VERSION:
            return None
        model = UNet().to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        return model
    except Exception:
        return None

def predict_star_positions(model, data):
    """Runs the trained model on a plate's raw pixel array, returns
    predicted star (x, y) positions."""
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)

    img_t = torch.tensor(img_8bit.astype(np.float32) / 255.0).unsqueeze(0)
    orig_h, orig_w = img_t.shape[1], img_t.shape[2]
    img_t = pad_to_multiple(img_t).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(img_t)
    pred = pred.squeeze().cpu().numpy()
    pred = pred[:orig_h, :orig_w]

    binary = (pred >= PRED_MASK_THRESHOLD).astype(np.uint8)
    labeled, n_obj = ndimage.label(binary)
    xs, ys = [], []
    for obj_id in range(1, n_obj + 1):
        region = labeled == obj_id
        if region.sum() < MIN_BLOB_AREA:
            continue
        coords = np.argwhere(region)
        cy, cx = coords.mean(axis=0)
        xs.append(float(cx)); ys.append(float(cy))
    return np.array(xs), np.array(ys)

# ============================================================================
# TARGET LOCATION -- same 3 tiers as 02_A_SISaP.ipynb, with a new PyTorch
# tier inserted between the APASS-list tiers and the direct-position
# fallback. Only reached when the target ISN'T already a bright,
# APASS-catalogued star at this epoch (i.e. exactly the faded/faint cases
# a fixed-threshold approach struggles with).
# ============================================================================

def locate_target(wcs, data, star_ra, star_dec, star_x, star_y, target_coord, model,
                   primary_max_sep=10, widened_max_sep=15):
    if len(star_ra) > 0:
        cat_coords = SkyCoord(np.asarray(star_ra) * u.deg, np.asarray(star_dec) * u.deg)
        sep = target_coord.separation(cat_coords)
        idx = int(np.argmin(sep))
        if sep[idx].arcsec < primary_max_sep:
            return float(star_x[idx]), float(star_y[idx]), "apass_list"
        if sep[idx].arcsec < widened_max_sep:
            return float(star_x[idx]), float(star_y[idx]), "apass_list_widened"

    try:
        tx, ty = wcs.world_to_pixel_values(target_coord.ra.deg, target_coord.dec.deg)
        tx, ty = float(np.array(tx)), float(np.array(ty))
    except Exception:
        return None, None, "not_found"

    margin = 5
    if not ((-margin <= tx < data.shape[1] + margin) and (-margin <= ty < data.shape[0] + margin)):
        return None, None, "not_found"

    # PyTorch tier: ask the trained model where the stars are, take
    # whichever predicted blob is closest to the expected position, if
    # any are within tolerance.
    if model is not None:
        try:
            pred_x, pred_y = predict_star_positions(model, data)
            if len(pred_x) > 0:
                d = np.hypot(pred_x - tx, pred_y - ty)
                j = int(np.argmin(d))
                if d[j] <= PYTORCH_MATCH_TOLERANCE_PX:
                    return float(pred_x[j]), float(pred_y[j]), "pytorch_prediction"
        except Exception:
            pass

    # Direct WCS position, centroid-refined.
    xr, yr = refine_centroids(data, np.array([tx]), np.array([ty]))
    if np.hypot(xr[0] - tx, yr[0] - ty) < CENTROID_BOX:
        x0, x1 = max(int(xr[0] - 10), 0), min(int(xr[0] + 11), data.shape[1])
        y0, y1 = max(int(yr[0] - 10), 0), min(int(yr[0] + 11), data.shape[0])
        if x1 > x0 and y1 > y0:
            local_patch = data[y0:y1, x0:x1]
            if local_patch.max() > np.nanpercentile(data, 80):
                return float(xr[0]), float(yr[0]), "direct_position"

    # Raw-pixel local-threshold blob (badly saturated/blooming target).
    try:
        search_r = 25
        x0, x1 = max(int(tx - search_r), 0), min(int(tx + search_r) + 1, data.shape[1])
        y0, y1 = max(int(ty - search_r), 0), min(int(ty + search_r) + 1, data.shape[0])
        if x1 > x0 and y1 > y0:
            local = data[y0:y1, x0:x1]
            local_thresh = np.nanpercentile(local, 90)
            bright_mask_local = local > local_thresh
            if bright_mask_local.any():
                ys_idx, xs_idx = np.nonzero(bright_mask_local)
                weights = local[ys_idx, xs_idx]
                cx_local = np.average(xs_idx, weights=weights)
                cy_local = np.average(ys_idx, weights=weights)
                return float(x0 + cx_local), float(y0 + cy_local), "raw_pixel_blob"
    except Exception:
        pass

    return None, None, "not_found"

# ============================================================================
# PER-PLATE PROCESSING (same structure as 02_A_SISaP.ipynb's process_plate,
# with locate_target() now taking the trained model as an extra argument)
# ============================================================================

def process_plate(fits_path, target_coord, model):
    try:
        data = astrofits.getdata(fits_path).astype(float)
        header = astrofits.getheader(fits_path)
        wcs = WCS(header)
        jd = get_plate_jd(header)
        data_sub, bkg_rms = subtract_background_2d(data)

        h, w = data.shape
        ra_c, dec_c = wcs.all_pix2world(w / 2, h / 2, 0)
        corners_x = [0, w - 1, 0, w - 1]
        corners_y = [0, 0, h - 1, h - 1]
        corner_ra, corner_dec = wcs.all_pix2world(corners_x, corners_y, 0)
        center = SkyCoord(float(ra_c) * u.deg, float(dec_c) * u.deg)
        corners_sky = SkyCoord(corner_ra * u.deg, corner_dec * u.deg)
        field_radius_arcsec = center.separation(corners_sky).max().arcsec * 1.05

        apass_stars = get_field_catalog(float(ra_c), float(dec_c), field_radius_arcsec)
        if len(apass_stars) < MIN_APASS_STARS_QUERY:
            return _empty_result(fits_path, jd, 'Too few APASS stars in field')

        ra = np.array([s[0] for s in apass_stars])
        dec = np.array([s[1] for s in apass_stars])
        apass_b = np.array([s[2] for s in apass_stars])

        x, y = wcs.all_world2pix(ra, dec, 0)
        margin = 20
        in_frame = (x > margin) & (x < w - margin) & (y > margin) & (y < h - margin)
        x, y, apass_b = x[in_frame], y[in_frame], apass_b[in_frame]

        if len(x) > REF_STARS_MAX_PER_PLATE:
            keep = np.random.choice(len(x), size=REF_STARS_MAX_PER_PLATE, replace=False)
            x, y, apass_b = x[keep], y[keep], apass_b[keep]

        if len(x) == 0:
            return _empty_result(fits_path, jd, 'No APASS stars in frame')

        x_ref, y_ref = refine_centroids(data_sub, x, y)
        isolated = filter_isolated_stars(x_ref, y_ref)
        flux, bkg_med, bkg_std = measure_aperture_photometry(data_sub, x_ref, y_ref)

        area = np.pi * DEFAULT_APERTURE**2
        with np.errstate(divide='ignore', invalid='ignore'):
            snr = np.where(bkg_std > 0, flux / (bkg_std * np.sqrt(area)), 0)

        is_sat = detect_saturation(data, x_ref, y_ref)
        with np.errstate(divide='ignore', invalid='ignore'):
            inst_mag = np.where(flux > 0, -2.5 * np.log10(np.maximum(flux, 1e-10)), np.nan)

        good_for_fit = (snr >= SIGNIF_THRESHOLD) & np.isfinite(inst_mag) & ~is_sat & isolated
        fit_inst_mag = inst_mag[good_for_fit]
        fit_apass_b = apass_b[good_for_fit]

        calibration_global = calibrate_global(fit_inst_mag, fit_apass_b)
        n_ref_used = calibration_global['n_used'] if calibration_global else int(good_for_fit.sum())

        target_x, target_y, target_tier = locate_target(
            wcs, data, ra[in_frame], dec[in_frame], x_ref, y_ref, target_coord, model
        )

        target_detected = False
        target_mag = np.nan
        target_mag_error = np.nan
        calibration_mode = None
        final_rms = np.nan
        final_n_used = 0
        tx_out, ty_out = (target_x, target_y) if target_x is not None else (np.nan, np.nan)

        if target_x is not None and calibration_global is not None:
            tx_ref, ty_ref = refine_centroids(data_sub, np.array([target_x]), np.array([target_y]))
            t_flux, t_bkg_med, t_bkg_std = measure_aperture_photometry(data_sub, tx_ref, ty_ref)
            t_area = np.pi * DEFAULT_APERTURE**2
            t_snr = t_flux[0] / (t_bkg_std[0] * np.sqrt(t_area)) if t_bkg_std[0] > 0 else 0
            tx_out, ty_out = float(tx_ref[0]), float(ty_ref[0])

            if t_snr >= SIGNIF_THRESHOLD and t_flux[0] > 0:
                t_inst_mag = -2.5 * np.log10(t_flux[0])
                rough_mag = invert_calibration(calibration_global, t_inst_mag)

                if rough_mag is not None:
                    calibration_local = calibrate_local(fit_inst_mag, fit_apass_b, rough_mag)
                    if calibration_local is not None:
                        local_mag = invert_calibration(calibration_local, t_inst_mag)
                        if local_mag is not None:
                            target_mag = local_mag
                            final_rms = calibration_local['rms']
                            final_n_used = calibration_local['n_used']
                            calibration_mode = 'local'
                    if calibration_mode is None:
                        target_mag = rough_mag
                        final_rms = calibration_global['rms']
                        final_n_used = calibration_global['n_used']
                        calibration_mode = 'global_fallback'

                    calibration_uncertainty = final_rms / np.sqrt(max(final_n_used, 1))
                    target_mag_error = np.sqrt((1.0857 / max(t_snr, 1e-6))**2 + calibration_uncertainty**2)
                    target_detected = True

        quality_ok = (
            calibration_global is not None and
            n_ref_used >= MIN_REF_STARS and
            target_detected and
            np.isfinite(final_rms) and
            final_rms <= MAX_CALIBRATION_RMS
        )

        rejection_reason = None
        if calibration_global is None:
            rejection_reason = 'Global calibration failed (insufficient good reference stars)'
        elif n_ref_used < MIN_REF_STARS:
            rejection_reason = f'Too few reference stars in fit ({n_ref_used} < {MIN_REF_STARS})'
        elif not target_detected:
            rejection_reason = f'Target not detected (location tier: {target_tier})'
        elif np.isfinite(final_rms) and final_rms > MAX_CALIBRATION_RMS:
            rejection_reason = f'Calibration RMS too high ({final_rms:.3f} > {MAX_CALIBRATION_RMS}, mode={calibration_mode})'

        result_row = {
            'filename': fits_path.name,
            'jd': jd,
            'target_detected': target_detected,
            'target_mag': target_mag,
            'target_mag_error': target_mag_error,
            'target_x': tx_out,
            'target_y': ty_out,
            'target_location_tier': target_tier,
            'num_reference_stars': n_ref_used,
            'zeropoint': calibration_global['coeffs'][-1] if calibration_global else np.nan,
            'rms_scatter': final_rms,
            'rms_scatter_global': calibration_global['rms'] if calibration_global else np.nan,
            'calibration_mode': calibration_mode,
            'quality_ok': quality_ok,
            'rejection_reason': rejection_reason,
        }
        return {'result_row': result_row}

    except Exception as e:
        return _empty_result(fits_path, np.nan, str(e))

def _empty_result(fits_path, jd, reason):
    return {
        'result_row': {
            'filename': fits_path.name, 'jd': jd, 'target_detected': False,
            'target_mag': np.nan, 'target_mag_error': np.nan,
            'target_x': np.nan, 'target_y': np.nan, 'target_location_tier': 'not_found',
            'num_reference_stars': 0, 'zeropoint': np.nan,
            'rms_scatter': np.nan, 'rms_scatter_global': np.nan,
            'calibration_mode': None, 'quality_ok': False,
            'rejection_reason': reason,
        },
    }

# ============================================================================
# PER-STAR RUNNER
# ============================================================================

def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def run_star(star_dir, target_coord, plate_progress_bar, plate_progress_html, plate_status_label):
    star_cutouts_dir = star_dir / 'cutouts'
    star_output_dir = star_dir / '03_A'
    star_output_dir.mkdir(parents=True, exist_ok=True)
    star_cache_dir = star_output_dir / 'algorithm_cache'
    star_cache_dir.mkdir(parents=True, exist_ok=True)

    results_path = star_output_dir / 'photometry_results.csv'
    lightcurve_path = star_output_dir / 'lightcurve.csv'

    cutouts = sorted(star_cutouts_dir.glob('*.fits'))
    total = max(len(cutouts), 1)

    # Train (or reuse) this star's own model, fully unattended.
    model = load_existing_model(star_dir)
    train_summary = None
    if model is None:
        plate_status_label.value = f"{star_dir.name}: training PyTorch model..."
        model, train_summary = build_dataset_and_train(star_dir, cutouts, plate_status_label)
    else:
        print(f"  {star_dir.name}: reusing previously-trained model (version {MODEL_VERSION}).")

    plate_progress_bar.max = total
    plate_progress_bar.value = 0

    plate_db = shelve.open(str(star_cache_dir / 'plate_db_shelf'), flag='c', writeback=False)

    start = time.monotonic()
    last_render = 0.0
    n_errors = 0
    tier_counts = {}

    for i, f in enumerate(cutouts):
        key = str(f)
        cached = plate_db.get(key)
        if cached and cached.get('_version', 0) == PIPELINE_VERSION and cached.get('_model_version') == MODEL_VERSION:
            pass
        else:
            plate_status_label.value = f"{star_dir.name}: {f.name}"
            out = process_plate(f, target_coord, model)
            if out['result_row'].get('num_reference_stars', 0) == 0:
                n_errors += 1
            out['_version'] = PIPELINE_VERSION
            out['_model_version'] = MODEL_VERSION
            plate_db[key] = out

        tier = plate_db[key]['result_row'].get('target_location_tier', 'not_found')
        tier_counts[tier] = tier_counts.get(tier, 0) + 1

        now = time.monotonic()
        if now - last_render > 0.15 or (i + 1) == total:
            elapsed = now - start
            pct = (i + 1) / total * 100
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            remaining = (total - i - 1) / rate if rate > 0 else None
            plate_progress_bar.value = i + 1
            plate_progress_html.value = (
                f"<div style='font-family: monospace; font-size: 12px;'>"
                f"<b>{pct:5.1f}%</b> &nbsp; {i+1:,}/{total:,} plates &nbsp;|&nbsp; "
                f"{rate:.2f} plates/sec &nbsp;|&nbsp; "
                f"Elapsed {format_eta(elapsed)} &nbsp;|&nbsp; ETA {format_eta(remaining)}"
                f"</div>"
            )
            last_render = now

        if (i + 1) % 50 == 0:
            plate_db.sync()

    plate_db.sync()

    result_rows = []
    for f_str in plate_db.keys():
        out = plate_db[f_str]
        if out is None:
            continue
        result_rows.append(out['result_row'])

    results_df = pd.DataFrame(result_rows).sort_values('jd').reset_index(drop=True)
    results_df.to_csv(results_path, index=False)

    lc_df = results_df[results_df['quality_ok']].copy()
    lc_df = lc_df.rename(columns={'target_mag': 'magnitude', 'target_mag_error': 'magnitude_error'})
    lc_df = lc_df[['jd', 'magnitude', 'magnitude_error']].dropna().sort_values('jd').reset_index(drop=True)

    n_before_clip = len(lc_df)
    if len(lc_df) > 20:
        window = 15
        rolling_med = lc_df['magnitude'].rolling(window, center=True, min_periods=5).median()
        residual = lc_df['magnitude'] - rolling_med
        clipped = sigma_clip(residual, sigma=4, maxiters=3)
        lc_df = lc_df[~clipped.mask].reset_index(drop=True)
    n_after_clip = len(lc_df)

    lc_df.to_csv(lightcurve_path, index=False)
    plate_db.close()

    return {
        'star': star_dir.name,
        'n_plates': len(results_df),
        'n_errors': n_errors,
        'n_lightcurve_points': n_after_clip,
        'n_removed_by_clip': n_before_clip - n_after_clip,
        'n_pytorch_tier': tier_counts.get('pytorch_prediction', 0),
        'model_trained': train_summary is not None,
        'output_dir': str(star_output_dir),
    }

# ============================================================================
# RUN ALL STARS
# ============================================================================

overall_progress_bar = widgets.IntProgress(value=0, min=0, max=len(star_dirs), description='Stars:')
overall_status_label = widgets.Label(value="Not started")
plate_progress_bar = widgets.IntProgress(value=0, min=0, max=1, description='Plates:')
plate_progress_html = widgets.HTML(value="")
plate_status_label = widgets.Label(value="")

display(widgets.VBox([
    widgets.HTML("<b>Overall progress across stars</b>"),
    widgets.HBox([overall_progress_bar, overall_status_label]),
    widgets.HTML("<b>Current star's training / plate progress</b>"),
    plate_status_label,
    plate_progress_bar,
    plate_progress_html,
]))

run_summaries = []
unresolved_stars = []

for i, star_dir in enumerate(star_dirs):
    overall_status_label.value = f"Resolving target for {star_dir.name}..."
    target_coord, coord_source = resolve_target_coord(star_dir)

    if target_coord is None:
        print(f"Could not resolve target coordinate for {star_dir.name} ({coord_source}) -- skipping. "
              f"Add it to MANUAL_TARGET_COORDS above and rerun.")
        unresolved_stars.append(star_dir.name)
        overall_progress_bar.value = i + 1
        continue

    overall_status_label.value = f"Processing {star_dir.name} ({i+1}/{len(star_dirs)}), coord source: {coord_source}"

    try:
        summary = run_star(star_dir, target_coord, plate_progress_bar, plate_progress_html, plate_status_label)
        run_summaries.append(summary)
        print(f"{star_dir.name}: {summary['n_plates']} plates processed, "
              f"{summary['n_lightcurve_points']} light curve points "
              f"({summary['n_pytorch_tier']} located via PyTorch tier) -> {summary['output_dir']}")
    except Exception as e:
        print(f"ERROR processing {star_dir.name}: {e}")
        traceback.print_exc()

    apass_cache.sync()
    overall_progress_bar.value = i + 1

overall_status_label.value = f"Done. Processed {len(run_summaries)} / {len(star_dirs)} stars."
apass_cache.sync()

print("\n" + "=" * 70)
print("MULTI-STAR PYTORCH RUN SUMMARY")
print("=" * 70)
summary_df = pd.DataFrame(run_summaries)
if len(summary_df) > 0:
    print(summary_df.to_string(index=False))
if unresolved_stars:
    print(f"\nSkipped (coordinate resolution failed): {unresolved_stars}")
    print("  Add these to MANUAL_TARGET_COORDS at the top of this cell and rerun.")


Found 14 star folders with a cutouts/ directory:
  R_CrB: 10555 plates
  RS_Tel: 5265 plates
  RY_Sgr: 7003 plates
  S_Aps: 6820 plates
  SU_Tau: 10272 plates
  U_Aqr: 8149 plates
  UW_Cen: 4771 plates
  V2552_Oph: 6806 plates
  V348_Sgr: 7084 plates
  V581_CrA: 5183 plates
  V854_Cen: 4542 plates
  V_CrA: 5405 plates
  XX_Cam: 10643 plates
  Y_Mus: 5760 plates

Loaded 0 cached APASS field queries (shared across all stars)


  R_CrB: 6849 plate(s) usable for training, 3706 skipped (too messy or empty).
  R_CrB: model trained (ran all 30 epochs, best val loss 0.0078). Saved to C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\R_CrB\03_A\pytorch_model.pth
ERROR processing R_CrB: name 'sigma_clip' is not defined


Traceback (most recent call last):
  File "C:\Users\dapur\AppData\Local\Temp\ipykernel_10072\2996641732.py", line 1083, in <module>
    summary = run_star(star_dir, target_coord, plate_progress_bar, plate_progress_html, plate_status_label)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dapur\AppData\Local\Temp\ipykernel_10072\2996641732.py", line 1029, in run_star
    clipped = sigma_clip(residual, sigma=4, maxiters=3)
              ^^^^^^^^^^
NameError: name 'sigma_clip' is not defined


  RS_Tel: 2482 plate(s) usable for training, 2783 skipped (too messy or empty).
  RS_Tel: model trained (stopped early at epoch 26, best val loss 0.0183). Saved to C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\RS_Tel\03_A\pytorch_model.pth


In [ ]:
# NOTE : 03_A_PyTorch.ipynb - Cell 2

# ============================================================================
# Reads each star's saved 03_A/lightcurve.csv and generates the
# This-PyTorch-Pipeline-vs-DASCH comparison plot -- same pattern as
# 02_A_SISaP.ipynb's Cell 2. Fully self-contained (rediscovers star
# folders from disk), can be run independently any time after Cell 1 has
# produced output files, even after a kernel restart.
# ============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DATA_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data")

star_dirs = sorted([d for d in BASE_DATA_DIR.iterdir()
                     if d.is_dir() and (d / 'cutouts').is_dir()])
print(f"Found {len(star_dirs)} star folders. Generating comparison plots...")

plot_summaries = []

for star_dir in star_dirs:
    lc_path = star_dir / '03_A' / 'lightcurve.csv'
    if not lc_path.exists():
        print(f"{star_dir.name}: no lightcurve.csv found (Cell 1 hasn't been run for this star yet) -- skipping.")
        continue

    lc_df = pd.read_csv(lc_path)

    # --- Fetch DASCH comparison for this star ---
    dasch_jd, dasch_mag, dasch_mag_err = None, None, None
    try:
        from daschlab import Session
        query_name = star_dir.name.replace('_', ' ')

        sess = Session(str(star_dir))
        sess.select_target(query_name)
        sess.select_refcat("apass")
        dlc = sess.lightcurve(0)

        dasch_jd_raw = np.asarray(dlc["time"].jd)
        dasch_mag_raw = np.asarray(dlc["magcal_magdep"].value)
        if "magcal_local_rms" in dlc.colnames:
            dasch_err_raw = np.asarray(dlc["magcal_local_rms"].value)
        else:
            dasch_err_raw = np.full_like(dasch_mag_raw, np.nan)

        good = np.isfinite(dasch_jd_raw) & np.isfinite(dasch_mag_raw)
        dasch_jd, dasch_mag = dasch_jd_raw[good], dasch_mag_raw[good]
        dasch_mag_err = dasch_err_raw[good]
    except Exception as e:
        print(f"  Could not fetch DASCH light curve for {star_dir.name}: {e}")

    # --- Combined comparison plot ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

    if len(lc_df) > 0:
        ax1.errorbar(lc_df['jd'], lc_df['magnitude'], yerr=lc_df['magnitude_error'],
                     fmt='o', color='#2ecc71', markersize=4,
                     ecolor='#16a085', elinewidth=1.0, capsize=1.5, capthick=0.6,
                     alpha=0.9, markeredgewidth=0)
        ax1.invert_yaxis()
    else:
        ax1.text(0.5, 0.5, 'No quality-passing plates', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_ylabel('B Magnitude')
    ax1.set_title(f'{star_dir.name} -- This PyTorch Pipeline ({len(lc_df)} points)')
    ax1.grid(True, alpha=0.3)

    if dasch_jd is not None and len(dasch_jd) > 0:
        ax2.errorbar(dasch_jd, dasch_mag, yerr=dasch_mag_err,
                     fmt='o', color='black', markersize=3,
                     ecolor='#e74c3c', elinewidth=0.6, capsize=1.2, capthick=0.5,
                     alpha=0.85, markeredgewidth=0)
        ax2.invert_yaxis()
        ax2.set_title(f'{star_dir.name} -- DASCH Reference (raw, {len(dasch_jd)} points)')
    else:
        ax2.text(0.5, 0.5, 'DASCH light curve unavailable', ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title(f'{star_dir.name} -- DASCH Reference (unavailable)')
    ax2.set_xlabel('Julian Date')
    ax2.set_ylabel('Calibrated Magnitude')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- Target-location tier breakdown (how often the PyTorch tier
    # actually contributed vs. the pre-existing APASS/direct/raw tiers) ---
    results_path = star_dir / '03_A' / 'photometry_results.csv'
    if results_path.exists():
        results_df = pd.read_csv(results_path)
        tier_counts = results_df['target_location_tier'].value_counts()
        print(f"  {star_dir.name} target-location tier breakdown:")
        print(tier_counts.to_string())

    plot_summaries.append({
        'star': star_dir.name,
        'n_pipeline_points': len(lc_df),
        'n_dasch_points': len(dasch_jd) if dasch_jd is not None else 0,
    })

print("\n" + "=" * 70)
print("PLOT GENERATION SUMMARY")
print("=" * 70)
summary_df = pd.DataFrame(plot_summaries)
if len(summary_df) > 0:
    print(summary_df.to_string(index=False))
